# Apache Spark DataFrames and Spark SQL

## A deep conceptual introduction

Apache Spark 3.5.9 · Structured processing · Catalyst · Spark Core execution

This notebook contains explanation slides only. The following notebooks apply these ideas with code.

# Learning objectives

By the end of this lesson, you should be able to:

- Explain what a Spark DataFrame represents.
- Compare DataFrames, SQL, and RDDs.
- Describe the journey from an API call or SQL statement to distributed execution.
- Explain the roles of Catalyst, statistics, physical planning, code generation, Spark Core, and storage systems.
- Recognize transformations, actions, partitions, shuffles, and common performance concerns.

# Why Spark needed a structured API

RDDs provide distributed collections and low-level transformations, but Spark usually cannot understand the meaning of arbitrary Python, Java, or Scala functions.

Structured workloads contain information Spark can reason about:

- Column names and data types
- Filters, projections, joins, and aggregations
- Relationships between expressions
- File formats, partitions, and statistics

DataFrames and Spark SQL expose this structure to Spark's optimizer.

# What is a DataFrame?

A DataFrame is an immutable, distributed collection of rows organized into named columns with a schema.

Conceptually, it resembles a table. Operationally, it is a description of a distributed computation that will produce rows.

A DataFrame normally contains:

- A schema
- A logical query plan
- References to input data or earlier plans
- A Spark session and execution context

It does not necessarily contain materialized result rows when it is created.

# Structured data

Structured data has a known organization that can be represented as fields and types.

| Field | Example type | Example value |
| --- | --- | --- |
| `movieId` | integer | `296` |
| `title` | string | `Pulp Fiction (1994)` |
| `rating` | double | `4.5` |
| `rated_at` | timestamp | `2026-08-26 09:30:00` |

Spark can also represent arrays, maps, nested structures, dates, decimals, and binary values.

# Rows, columns, and schema

A **row** is one record. A **column** is a typed expression evaluated for every relevant row. A **schema** describes the column names, data types, and nullability.

The schema enables Spark to:

- Resolve column references
- Check whether expressions are valid
- Select appropriate functions and encoders
- Estimate storage and execution requirements
- Read and write data consistently

Nullability is part of the schema, but source-data quality must still be validated.

# DataFrames are immutable

A transformation does not modify its input DataFrame. It returns another DataFrame with a new logical plan.

```text
source DataFrame
      │
      ├── filter ──> filtered DataFrame
      │
      └── select ──> projected DataFrame
```

The DataFrames can share parts of the same plan. Immutability makes lineage, optimization, and recomputation possible.

# Two main structured code flavors

Spark offers two common ways to describe the same type of work:

**DataFrame API flavor**

```python
ratings.groupBy("movieId").avg("rating")
```

**SQL flavor**

```sql
SELECT movieId, AVG(rating)
FROM ratings
GROUP BY movieId
```

These are illustrative expressions, not cells to execute in this notebook.

# DataFrame API and SQL share one engine

The DataFrame API and SQL syntax enter the same Spark SQL engine.

```text
DataFrame expressions ─┐
                      ├─> logical plans ─> Catalyst ─> physical plan
SQL text ─> SQL parser ┘
```

SQL is not a separate execution system. After parsing and analysis, both styles are represented by Spark's internal plan structures and can receive comparable optimizations.

# Choosing a code flavor

Use the **DataFrame API** when:

- Building a programmatic pipeline
- Reusing language variables and functions
- Composing transformations dynamically
- Integrating structured work with application logic

Use **SQL** when:

- Expressing relational analysis clearly
- Working with analysts familiar with SQL
- Reusing established SQL patterns
- Querying registered views or catalog tables

Real projects frequently use both in the same pipeline.

# SparkSession: the structured entry point

`SparkSession` is the main entry point for DataFrames and Spark SQL. It coordinates:

- DataFrame creation and data-source access
- SQL parsing and query execution
- Runtime SQL configuration
- Catalogs, databases, tables, and views
- User-defined functions
- Access to the underlying `SparkContext`

The `SparkContext` represents the connection to Spark Core. A process should normally have only one active SparkContext.

# Data sources become logical relations

Spark can build DataFrames from CSV, JSON, Parquet, ORC, JDBC, Hive tables, and many other sources.

When a source is read, Spark records a relation in the logical plan. File content is normally read later, when an action requires results.

The data source supplies capabilities such as:

- Schema discovery or enforcement
- Column pruning
- Filter pushdown
- Partition discovery
- Batch or streaming reads
- Statistics, when available

# Explicit schema versus inference

An **explicit schema** is usually preferable for stable pipelines because it:

- Documents the expected contract
- Avoids or reduces inference work
- Prevents accidental type changes
- Makes malformed input easier to detect
- Produces predictable downstream behavior

Schema inference is convenient for exploration, but it may require extra source scans and can infer an unsuitable type from limited data.

# Transformations and actions

Structured transformations include selecting, filtering, joining, grouping, sorting, and adding columns. They build plans lazily.

Actions request a result or an external effect. Examples include:

- Displaying or collecting rows
- Counting rows
- Writing a DataFrame
- Executing a SQL command that changes a table

An action gives Spark the complete plan it needs to analyze, optimize, and execute.

# Lazy evaluation

Spark delays execution so it can inspect a complete chain of operations.

```text
read → filter → select → group → aggregate → action
        no distributed work yet             execution begins
```

Because Spark sees the whole plan, it may reorder, combine, simplify, or remove work while preserving the query's meaning.

# The structured query journey

```text
DataFrame API or SQL text
          ↓
Unresolved logical plan
          ↓
Analyzed logical plan
          ↓
Optimized logical plan
          ↓
Candidate physical plans
          ↓
Selected physical plan
          ↓
Stages and tasks on Spark executors
          ↓
Result, table, or files
```

Each stage answers a different question: meaning, validity, optimization, implementation, and distributed execution.

# SQL parsing

SQL text must first be parsed into an unresolved logical plan.

The parser recognizes syntax such as:

- `SELECT`, `FROM`, and `WHERE`
- Joins and subqueries
- Aggregations and window functions
- Data-definition and data-manipulation statements

DataFrame calls do not require SQL-text parsing, but they also create unresolved or partially resolved logical-plan expressions.

# Analysis and name resolution

The analyzer uses catalog and schema information to resolve meaning.

It determines:

- Which table or view a name refers to
- Which input provides each column
- Whether functions exist and accept the supplied types
- Whether implicit type coercion is valid
- Whether an expression is ambiguous or unresolved

Missing columns, ambiguous join columns, and incompatible expressions normally fail during analysis—before tasks run.

# Catalyst optimizer

Catalyst is Spark SQL's extensible framework for representing and transforming query plans.

It uses tree structures and transformation rules to move from unresolved expressions toward executable plans. Catalyst covers more than optimization: parsing, analysis, logical optimization, and physical planning all participate in the structured query pipeline.

The important idea is that Spark reasons about the computation before distributing it.

# Examples of logical optimization

Catalyst may apply rules such as:

- **Predicate pushdown:** move filters closer to the source.
- **Column pruning:** read or carry only required columns.
- **Constant folding:** evaluate constant expressions once.
- **Boolean simplification:** remove redundant conditions.
- **Null propagation:** simplify expressions using null semantics.
- **Projection collapsing:** combine compatible selections.

The exact rules applied depend on the plan and Spark version.

# Data-source pushdown

Some optimizations cross the boundary into a storage system.

```text
Query needs: movieId, rating WHERE rating >= 4
                      ↓
Parquet reader may read only two columns
and skip row groups using stored statistics
```

Pushdown capabilities vary by source. A JDBC database, Parquet file, CSV file, and custom connector do not offer identical pruning or filtering behavior.

# Statistics and cost-based decisions

Logical equivalence does not mean equal cost. Spark can use statistics such as row counts, estimated sizes, distinct counts, and column distributions.

Statistics help the optimizer choose among alternatives, particularly join order and join strategy. Missing or stale statistics can lead to weaker decisions.

Rule-based optimization is always important; cost-based optimization adds estimates where enough information is available.

# Physical planning

A logical plan states **what** result is required. A physical plan states **how** Spark intends to produce it.

Physical choices include:

- Scan implementation
- Hash, sort-merge, nested-loop, or broadcast join
- Partial and final aggregation
- Exchange operations for repartitioning
- Sort placement
- Row-based or columnar execution transitions

Spark compares valid strategies and selects an executable plan.

# Adaptive Query Execution

Adaptive Query Execution (AQE) can revise parts of a physical plan using statistics collected while the query runs.

AQE may:

- Coalesce small shuffle partitions
- Handle skewed shuffle partitions
- Change a join strategy when runtime sizes differ from estimates
- Improve partition sizing after an exchange

This means the final executed plan can differ from the initial physical plan.

# Whole-stage code generation

For compatible operators, Spark SQL can combine parts of a physical plan and generate JVM bytecode for a processing pipeline.

This reduces overhead from:

- Repeated virtual function calls
- Creating unnecessary intermediate objects
- Moving rows through one operator at a time

Not every operator or expression participates in the same generated pipeline. Boundaries can appear around exchanges, external code, and unsupported operators.

# Internal row representation

User-facing rows are convenient objects with named fields. During execution, Spark SQL commonly uses compact internal representations designed for efficient JVM processing.

This enables:

- Lower object overhead
- Efficient binary comparison and copying
- Generated access to known fields
- Better memory and cache behavior

The schema tells Spark how to interpret these internal values correctly.

# Where Spark Core and RDDs fit

Spark SQL does not replace Spark Core. It uses the core distributed runtime for:

- Applications, drivers, and executors
- Jobs, stages, and tasks
- Scheduling and resource use
- Partitions and shuffle transfer
- Caching, fault recovery, and locality

Physical Spark SQL operators ultimately run distributed work through Spark's execution infrastructure, historically and internally built on RDD abstractions.

# Important nuance: “DataFrame becomes an RDD”

It is useful but incomplete to say that a DataFrame or SQL query is “converted to an RDD.”

More accurately:

1. The structured expression becomes logical and physical query plans.
2. Spark SQL operators use optimized internal row or columnar processing.
3. Spark Core schedules the resulting partitioned computation as stages and tasks.
4. RDD abstractions remain part of the execution foundation, but the optimizer does not merely rewrite the query as the RDD code a user would have written.

Calling `.rdd` explicitly crosses into the lower-level public RDD API and may give up structured optimization opportunities after that boundary.

# DataFrame versus RDD

| Concern | DataFrame / SQL | RDD |
| --- | --- | --- |
| Data model | Typed schema and named columns | Language objects or tuples |
| Optimization | Catalyst can inspect expressions | User functions are mostly opaque |
| Execution | Spark SQL operators and code generation | RDD transformation functions |
| Storage optimizations | Pushdown and column pruning | Usually handled manually or by input APIs |
| Best fit | Structured and semi-structured analytics | Low-level or unstructured processing |

RDDs remain useful, but DataFrames are normally the first choice for structured workloads.

# Partitions remain fundamental

A DataFrame is divided into partitions. Each task processes one partition for a stage.

Input partitions can be influenced by:

- File sizes and file boundaries
- HDFS blocks and split rules
- Source-specific partitioning
- Explicit repartitioning
- Shuffle partition configuration

Too few partitions limit parallelism. Too many tiny partitions increase scheduling, metadata, and file-management overhead.

# Narrow operations and shuffles

A narrow operation can obtain each output partition from a small number of input partitions without redistributing all records.

A shuffle redistributes data across executors, commonly for:

- Grouping and aggregation by key
- Many joins
- Global sorting
- Deduplication
- Explicit repartitioning

Shuffles create exchange boundaries, often separate stages, and involve network, disk, serialization, and memory costs.

# Joins are a major physical decision

The same logical join can have different physical implementations.

- **Broadcast hash join:** send a small side to executors and avoid a large two-sided shuffle.
- **Sort-merge join:** shuffle compatible keys and process sorted partitions.
- **Shuffle hash join:** build partition-local hash structures after redistribution.
- **Nested-loop strategies:** needed for certain non-equality joins or special cases.

Data size, join condition, statistics, hints, and configuration affect the choice.

# Aggregation is often multi-stage

Spark commonly performs partial aggregation before shuffling, then combines partial results after records with the same grouping key meet.

```text
partition-local partial aggregates
              ↓
      shuffle by group key
              ↓
        final aggregates
```

Reducing data before the shuffle can substantially lower network traffic. Highly skewed keys can still overload individual partitions.

# Caching and persistence

Caching can avoid recomputation when the same expensive DataFrame is reused by multiple actions.

Cache selectively when:

- The plan is expensive to reproduce.
- The result is reused.
- The cached size fits the chosen storage level.

Caching every intermediate result wastes executor memory and may make performance worse. Materialization occurs only after an action, and cached data should be unpersisted when no longer needed.

# Python and the JVM boundary

Spark's scheduler and Spark SQL engine run in the JVM. PySpark provides a Python interface to them.

Built-in DataFrame expressions are described from Python but generally executed by optimized JVM Spark SQL operators. Ordinary Python UDFs may require data transfer between the JVM and Python workers and can restrict optimization or code generation.

Prefer built-in SQL functions when they express the required logic. Use UDFs when necessary and understand their execution cost.

# Tables, views, and catalogs

- A **temporary view** gives a DataFrame a SQL name within a session.
- A **global temporary view** has broader application scope but is still temporary.
- A **table** has persistent metadata in a catalog.
- A **managed table** lets the catalog manage both metadata and its table location.
- An **external table** keeps metadata in the catalog while data lifecycle remains external.

The Hive catalog allows Spark and compatible tools to discover shared databases and tables.

# DataFrame data is not necessarily stored in Spark

A DataFrame may represent data that currently resides in:

- HDFS, S3, or another object store
- Local or distributed files
- A relational database
- A Hive table
- An in-memory collection
- A streaming source
- The output of another logical plan

Spark reads partitions when execution requires them. A DataFrame is an interface and plan, not a storage format.

# Fault tolerance and lineage

Spark tracks how partitions are derived. If an executor loses a partition, Spark can normally recompute it from the source and lineage rather than requiring every intermediate result to be replicated.

Recovery may involve:

- Re-reading source partitions
- Re-running earlier tasks
- Re-fetching or regenerating shuffle data
- Recomputing cached partitions

External side effects require special care because re-executed tasks must not corrupt results.

# Reading an execution plan

An explained query plan commonly exposes several plan levels. Look for:

- Scans and pushed filters
- Projected columns
- Exchanges, which indicate redistribution
- Join algorithms
- Partial and final aggregates
- Sorts
- Adaptive plan markers
- Estimated or runtime statistics

Plans are evidence. Use them to confirm whether Spark is doing what you expect.

# Common misconceptions

**“A DataFrame is an in-memory table.”**  
It is usually a lazy plan; its data may remain in an external source.

**“SQL is slower than the DataFrame API.”**  
Both use the same planning engine; equivalent expressions can produce equivalent plans.

**“Every transformation starts a Spark job.”**  
Transformations are normally lazy; actions trigger execution.

**“One DataFrame equals one RDD.”**  
Structured plans and their physical execution are more nuanced than a one-to-one public RDD mapping.

# Practical design guidance

- Define schemas for stable inputs.
- Select required columns early.
- Filter early when it preserves meaning.
- Use built-in functions before writing UDFs.
- Watch for shuffles, skew, and accidental cross joins.
- Do not collect large datasets to the driver.
- Choose partition counts appropriate to data and resources.
- Prefer columnar formats such as Parquet or ORC for analytics.
- Inspect plans and measure real workloads instead of relying on folklore.

# Mental model to retain

```text
Structured intent
DataFrame expressions or SQL
            ↓
Spark SQL understands schema and meaning
            ↓
Catalyst analyzes and optimizes a query plan
            ↓
A physical plan selects executable operators
            ↓
Spark Core schedules partitioned stages and tasks
            ↓
Executors read, transform, shuffle, and write data
```

DataFrames provide information. Catalyst uses that information. Spark Core performs the distributed work.

# Review questions

1. Why can Spark optimize a DataFrame expression more deeply than an arbitrary RDD function?
2. What is the difference between a logical plan and a physical plan?
3. Why does an action matter to lazy evaluation?
4. Where do Spark Core and RDD abstractions fit beneath Spark SQL?
5. Why is “a DataFrame is converted directly into an RDD” an incomplete explanation?
6. What operations are likely to introduce a shuffle?
7. When would a temporary view be insufficient?

# Next lessons

Continue with the practical notebooks in this directory:

1. `S052-DataFrameBasic.ipynb` — creating and transforming DataFrames
2. `S052-DFJoin.ipynb` — join behavior and physical planning
3. `S053-MovieLens.ipynb` — DataFrame analytics with HDFS
4. `S057-SparkDatabaseHDFS.ipynb` — persistent tables and Hive catalog
5. `S059-MovieLens-SQL (1).ipynb` — SQL and DataFrame interoperability